# SABRE: Simulation of the coherent dynamics
This spin dynamics simulation demonstrates the coherent evolution at level anti-crossings under a static polarisation transfer field $B_0$.

## Details
- Spin system: Three spins (hydride protons + $^{19}\mathrm{F}$ from 3-fluoropyridine)
- Only coherent dynamics are simulated (no chemical exchange or relaxation)
- Populations of the eight states in singlet-triplet Zeeman basis are followed

### Imports
The simulations are performed using the Spinguin (v. 0.0.1) package. In addition, we use the functionality from NumPy and Matplotlib for creating arrays and plotting, respectively.

In [ ]:
import spinguin as sg
import numpy as np
import matplotlib.pyplot as plt

### Build the required spin systems
We build our three-spin system that consists of the hydride protons and the $^{19}\mathrm{F}$ from the 3-fluoropyridine ligand. In addition, we construct spin systems for the hydride protons and the $^{19}\mathrm{F}$ nucleus separately in order to conveniently create the coupled states. The full Liouville space basis sets are constructed for each of the system.

In [ ]:
ss_c = sg.SpinSystem(["1H", "1H", "19F"])
ss_c.basis.max_spin_order = 3
ss_c.basis.build()

ss_H2 = sg.SpinSystem(["1H", "1H"])
ss_H2.basis.max_spin_order = 2
ss_H2.basis.build()

ss_s = sg.SpinSystem(["1H"])
ss_s.basis.max_spin_order = 1
ss_s.basis.build()

### Define chemical shifts and $J$-couplings
We set the chemical shifts in ppm and the $J$-couplings in Hz.

In [ ]:
ss_c.chemical_shifts = [-23.1, -23.1, -124.2]

ss_c.J_couplings = [
    [0,     0,      0],
    [-7.67, 0,      0],
    [-0.05, 2.08,   0]
]

### Build the states for hydride protons and the $^{19}\mathrm{F}$
Next, we build the singlet, triplet-plus, triplet-zero, and triplet-minus states for the hydride protons and the alpha and beta states for the $^{19}\mathrm{F}$. These are used to construct the three-spin states in the next step.

In [ ]:
S = sg.singlet_state(ss_H2, 0, 1)
Tp = sg.triplet_plus_state(ss_H2, 0, 1)
T0 = sg.triplet_zero_state(ss_H2, 0, 1)
Tm = sg.triplet_minus_state(ss_H2, 0, 1)

a = sg.alpha_state(ss_s, 0)
b = sg.beta_state(ss_s, 0)

### Combine the states to get the detection states
We combine the previously built states using the associate() function in order to get all the eight states in the singlet-triplet Zeeman representation. These entities will be used to measure the populations of the states later.

In [ ]:
spin_map_H2 = [0, 1]
spin_map_s = [2]

Sa = sg.associate(ss_H2, ss_s, ss_c, S, a, spin_map_H2, spin_map_s)
Tpb = sg.associate(ss_H2, ss_s, ss_c, Tp, b, spin_map_H2, spin_map_s)
T0a = sg.associate(ss_H2, ss_s, ss_c, T0, a, spin_map_H2, spin_map_s)

Sb = sg.associate(ss_H2, ss_s, ss_c, S, b, spin_map_H2, spin_map_s)
Tma = sg.associate(ss_H2, ss_s, ss_c, Tm, a, spin_map_H2, spin_map_s)
T0b = sg.associate(ss_H2, ss_s, ss_c, T0, b, spin_map_H2, spin_map_s)

Tpa = sg.associate(ss_H2, ss_s, ss_c, Tp, a, spin_map_H2, spin_map_s)
Tmb = sg.associate(ss_H2, ss_s, ss_c, Tm, b, spin_map_H2, spin_map_s)

detection = [Sa, Tpb, T0a, Sb, Tma, T0b, Tpa, Tmb]
detection_str = [r"$S\alpha$", r"$T_+\beta$", r"$T_0\alpha$",
                 r"$S\beta$", r"$T_-\alpha$", r"$T_0\beta$",
                 r"$T_+\alpha$", r"$T_-\beta$"]

### Create the time evolution propagator
Next, we set the static magnetic field of $B_0 = 3.26\ \mathrm{\mu T}$ (the LAC condition) and the time step of $dt = 1\ \mathrm{ms}$. Then, we build the Hamiltonian, Liouvillian, and the time propagator.

In [ ]:
sg.parameters.magnetic_field = 3.26e-6
dt = 1e-3

H = sg.hamiltonian(ss_c)
L = sg.liouvillian(H)
P = sg.propagator(L, dt)

### Time evolution
We simulate the spin system for a short time of $1\ \mathrm{s}$ just to illustrate the coherent oscillation. The populations, which are acquired using the detection states during each time step, are stored in an array for further analysis.

In [ ]:
time = 1
nsteps = int(time/dt)
rho_c = sg.singlet_state(ss_c, 0, 1)
populations = np.zeros((nsteps, len(detection)))

for i in range(nsteps):

    for j in range(len(detection)):
        populations[i, j] = (detection[j].conj().T @ rho_c).trace().real * 100

    rho_c = P @ rho_c

### Plot the populations
The coherent evolution of the states are plotted. We notice the interchange of singlet-beta and triplet-plus alpha whereas the other states remain somewhat constant.

In [ ]:
t = np.linspace(0, nsteps*dt, nsteps, endpoint=False)
for i in range(len(detection)):
    plt.plot(t, np.real(populations[:, i]), label=detection_str[i])
plt.legend(loc="upper right")
plt.xlabel("Time (s)")
plt.ylabel("Population (%)")
plt.tight_layout()
plt.show()
plt.clf()

### Save the populations
Finally, the populations are saved as CSV for further analysis.

In [ ]:
np.savetxt(f"populations.csv", populations, delimiter=",")